# MS2 scoring: one call scores every candidate annotation

**Two files in, every candidate annotation scored.** The spectrum and the list of
possible annotations (with their SMILES) go in; each candidate comes back with the share
of the measured MS2 signal its theoretical fragments can explain.

This notebook is the **application**: settings, one call, results. The step-by-step
walkthrough of the method - cut -> cap -> mass -> theoretical m/z -> peak matching, and
the neutral-loss analysis behind Task 2 - lives in `ms2_scoring_stepwise.ipynb`.


In [1]:
import ms2_scoring as bf


## 1. Input

The **two files are the only required inputs**; everything else is a keyword argument of
`bf.explain_spectrum` and is set once, here.

- **`SPECTRUM`** - the fragmentation spectrum (`mz`, `rel_intensity`, `precursor_mz`).
- **`ANNOTATIONS`** - the possible annotations with their `smiles` (+ `name`, `id`, `adduct`).

  Any pair of files with those columns works. Column names are matched case-insensitively
  through a synonym table, and anything that has to be filled in (e.g. `rel_intensity`
  computed from `intensity`) is reported in `R["input_notes"]`, never done silently.
- **`METHOD`** / **`N`** - `"FragmentOnBonds"` cuts every combination of 1..N BRICS bonds
  (keeps the large diagnostic fragments); `"BRICSDecompose"` is the tutorial's full
  decomposition. `N` applies to the first only.
- **`PPM`** / **`MIN_DA`** - matching window = `max(MIN_DA, m/z * PPM)`. At low mass
  `MIN_DA` is usually the one that decides.
- **`MIN_REL_INT`** - noise floor in % relative intensity.
- **`DROP_ABOVE_PRECURSOR`** - drop peaks at or above the precursor m/z: a product ion
  cannot outweigh the ion it came from, so those are the surviving precursor, its 13C
  isotopes or a co-isolated contaminant. They also leave the score's denominator, so the
  score measures the fragment signal only.


In [2]:
# ===== the two INPUT FILES =====
SPECTRUM    = r"../data/example_features/FT6080_ms2_spectrum.csv"
ANNOTATIONS = r"../data/example_features/FT6080_ipa_annotations.csv"

# the other example feature (FT4679, a PC) - swap the two lines above for these:
# SPECTRUM    = r"../data/example_features/FT4679_ms2_spectrum.csv"
# ANNOTATIONS = r"../data/example_features/FT4679_ipa_annotations.csv"

# ===== settings =====
METHOD  = "FragmentOnBonds"   # "FragmentOnBonds" / "BRICSDecompose"
N       = 2                   # FragmentOnBonds: cut up to N bonds
PPM     = 20                  # m/z matching tolerance in ppm
MIN_DA  = 0.005               # absolute floor of the match window, in Da
MIN_REL_INT = 1.0             # noise floor, % relative intensity
DROP_ABOVE_PRECURSOR = True   # drop peaks at / above the precursor m/z


## 2. Run

One call reads both files, filters the peaks, fragments every candidate and scores it.
It returns the **result bundle** `R`, and everything below reads from it:

| key | what it holds |
| --- | --- |
| `R["scores"]` | the scoreboard, one row per annotation, best first |
| `R["candidates"]` | the full per-candidate results |
| `R["explained"]` | per annotation, one row per peak saying what explains it |
| `R["ann"]`, `R["peaks"]`, `R["precursor_mz"]`, `R["adduct"]` | the data as read and filtered |
| `R["method"]`, `R["ppm"]`, `R["min_da"]`, ... | the settings **actually** used |
| `R["dropped"]`, `R["input_notes"]` | what the filters removed, and any fallback applied |


In [3]:
R = bf.explain_spectrum(SPECTRUM, ANNOTATIONS, METHOD, N, PPM, MIN_DA,
                        MIN_REL_INT, DROP_ABOVE_PRECURSOR)

R["ann"][["name", "formula", "adduct", "smiles"]]    # the candidates that were scored


[adducts] 6 loaded from web (github raw): M+H, M+, M+Na, M+2H, 2M+H, M+NH4


spectrum   : FT6080_ms2_spectrum.csv
annotations: FT6080_ipa_annotations.csv (7 candidates)
settings   : FragmentOnBonds | N = 2 | ppm = 20 | min_da = 0.005
peaks      : 125 raw -> 56 kept (2 at/above the precursor, 67 below the noise floor)
precursor  : 872.7541 [M+NH4]
best score : 66.3% - TG(16:1(9Z)/18:1(9Z)/18:2(9Z,12Z))


,name,formula,adduct,smiles
0,"TG(16:0/16:0/20:4(5Z,8Z,11Z,14Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCCCCCCCCCC)(COC(=O)CCC\C=C...
1,"TG(18:1(11Z)/16:0/18:3(9Z,12Z,15Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCCCC\C=C/CCCCCC)(COC(=O)CC...
2,"TG(18:1(11Z)/16:1(9Z)/18:2(9Z,12Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCCCC\C=C/CCCCCC)(COC(=O)CC...
3,"TG(16:1(9Z)/18:1(9Z)/18:2(9Z,12Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCC\C=C/CCCCCC)(COC(=O)CCCC...
4,"TG(16:0/18:2(9Z,12Z)/18:2(9Z,12Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCCCCCCCCCC)(COC(=O)CCCCCCC...
5,"TG(18:1(9Z)/16:0/18:3(9Z,12Z,15Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCC\C=C/CCCCCCCC)(COC(=O)CC...
6,"TG(16:1(9Z)/18:1(11Z)/18:2(9Z,12Z))",C55H102NO6,M+NH4,[H][C@](COC(=O)CCCCCCC\C=C/CCCCCC)(COC(=O)CCCC...


## 3. The score of each annotation

**One number per candidate: the share of the measured MS2 signal it can explain.**

```
score = sum(rel_int of every peak this candidate explains, Task 1 or Task 2)
        -------------------------------------------------------------------  x 100
                        sum(rel_int of all peaks)
```

Both sums run over the filtered peak list, i.e. **after the peaks at or above the
precursor have been removed** - the score is a share of the fragment signal, not of the
precursor. A peak counts as explained when the candidate accounts for it **because of
its own structure**:

- **Task 1** - it matched a theoretical BRICS fragment ion (or is a 13C isotope of one);
- **Task 2, diagnostic** - a fatty-acid or head-group loss links it, naming a real
  structural part of *this* molecule.

The two sets are disjoint (Task 2 only looks at what Task 1 left over), so nothing is
counted twice.

**The generic Task-2 tier is deliberately not counted** (greyed out, next to
`unexplained_pct`). Generic losses - H2O, CO, CH2, C2H4 ... - come from a universal
library of 9 small neutrals, so which peaks they reach does not depend on the candidate:
measured on FT6080, the set of universal-reachable peaks is bit-for-bit identical for all
7 candidates. A weaker candidate does not explain fewer peaks, its peaks are merely
relabelled from `task1`/`diagnostic` to `generic` - count them and **every candidate
lands on the same 96.2%**, and the ranking disappears. That all-inclusive figure is a
fair statement about the *method's* reach, and useless for comparing annotations.


In [4]:
bf.style_scores(R["candidates"])      # greyed columns are NOT counted in the score


,name,id,score,task1_pct,task2_diagnostic_pct,task2_generic_pct,unexplained_pct,n_explained,n_peaks
0,"TG(16:1(9Z)/18:1(9Z)/18:2(9Z,12Z))",HMDB0005440,66.3,46.9,19.5,29.9,33.7,19,56
1,"TG(18:1(11Z)/16:1(9Z)/18:2(9Z,12Z))",HMDB0010444,66.1,46.6,19.5,30.1,33.9,18,56
2,"TG(16:1(9Z)/18:1(11Z)/18:2(9Z,12Z))",HMDB0048618,66.1,46.6,19.5,30.1,33.9,18,56
3,"TG(18:1(11Z)/16:0/18:3(9Z,12Z,15Z))",HMDB0010440,65.5,45.4,20.1,30.7,34.5,17,56
4,"TG(18:1(9Z)/16:0/18:3(9Z,12Z,15Z))",HMDB0010451,64.6,44.5,20.1,31.6,35.4,17,56
5,"TG(16:0/16:0/20:4(5Z,8Z,11Z,14Z))",HMDB0005363,64.0,46.2,17.8,32.2,36.0,16,56
6,"TG(16:0/18:2(9Z,12Z)/18:2(9Z,12Z))",HMDB0005390,62.6,42.4,20.2,33.7,37.4,14,56


## 4. The interactive report

One self-contained page, four quadrants - no server, no internet, no live kernel:

| | |
| --- | --- |
| **top-left** | the two input files and the settings this run used |
| **top-right** | the mirror spectrum: measured peaks above, the selected candidate's theoretical fragment ions below |
| **bottom-left** | the structure of the selected candidate |
| **bottom-right** | the candidates, best score first - **click a row** to load it into the other three |

**Peak colours** say what the method did with each peak:

<span style="color:#b9bfc7">&#9632;</span> **grey** - removed before the analysis (below the noise floor, or at/above the precursor) &nbsp;
<span style="color:#2f9e44">&#9632;</span> **green** - a theoretical fragment matched it &nbsp;
<span style="color:#1a56c4">&#9632;</span> **blue** - a neutral loss explains it (pale blue = generic loss, not counted in the score) &nbsp;
<span style="color:#e0524a">&#9632;</span> **red** - nothing explains it

The coloured-in area is the score made visible: green + dark blue is exactly what
section 3 adds up.

**Click any peak** for a popup - a green one names the fragment (formula, adduct,
theoretical m/z, ppm error) and draws its structure; a blue one shows the loss that
formed it; a grey one says why it was dropped. Theoretical sticks below the axis are
clickable too, including the ones that were predicted but never observed. Drag across
the plot to zoom into an m/z range, double-click to reset.

`save=True` also writes a standalone `<feature>_report.html` next to this notebook - that
single file is the thing to send to someone else.


In [5]:
bf.spectrum_app(R, save=True)     # shown here, and written as a standalone .html


wrote .\FT6080_report.html  (415 KB)


D:\miniconda3\envs\msfrag\Lib\site-packages\IPython\core\display.py:452: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### The table behind the figure

`R["explained"][name]` is what the report is drawn from - one row per peak, with what
explains it and whether the score counts it. `counted` is the link between the picture
and the number: those intensities are exactly the score, so a peak can never look
explained in the figure and be missing from section 3.


In [6]:
et = R["explained"][R["best"]]                     # the best-scoring annotation
print("kinds:", et["kind"].value_counts().to_dict())
print("counted intensity: %.1f%%  ==  score %.1f%%"
      % (100 * et.loc[et["counted"], "rel_int"].sum() / sum(r for _, r in R["peaks"]),
         R["scores"]["score"].iloc[0]))
et.head(15)


kinds: {'generic': 29, 'fragment': 13, '': 8, 'diagnostic': 6}
counted intensity: 66.3%  ==  score 66.3%


,exp_mz,rel_int,kind,counted,detail,frag_smiles,name,formula,best_adduct,theo_mz,ppm,n_hits
0,575.5035,100.000,fragment,True,diacylglycerol (DAG) [M+H] 575.5034 (+0.3 ppm),CCCCC/C=C\C/C=C\CCCCCCCC(=O)OCCCOC(=O)CCCCCCC/...,diacylglycerol (DAG),C37H66O4,M+H,575.5034,0.3,1
1,601.5189,94.046,diagnostic,True,acyl bridge: precursor - 271.251 (NH3 + fatty ...,,,,,NaN,NaN,0
2,577.5189,83.674,fragment,True,diacylglycerol (DAG) [M+H] 577.5190 (-0.2 ppm),CCCCCC/C=C\CCCCCCCC(=O)OC[C@@H](C)OC(=O)CCCCCC...,diacylglycerol (DAG),C37H68O4,M+H,577.5190,-0.2,1
3,67.0544,23.211,fragment,True,C3H8 [M+Na] 67.0518 (+37.8 ppm),CCC,,C3H8,M+Na,67.0518,37.8,1
4,81.0699,19.751,generic,False,small loss: 111.080-81.070=30.010 (formaldehyde),,,,,NaN,NaN,0
5,95.0855,18.023,generic,False,series spacing: 123.117-95.085=28.031 (ethylene),,,,,NaN,NaN,0
6,69.0700,17.964,generic,False,small loss: 97.065-69.070=27.995 (carbon monox...,,,,,NaN,NaN,0
7,603.5343,11.020,fragment,True,diacylglycerol (DAG) [M+H] 603.5347 (-0.6 ppm),CCCCC/C=C\C/C=C\CCCCCCCC(=O)OC[C@H](C)OC(=O)CC...,diacylglycerol (DAG),C39H70O4,M+H,603.5347,-0.6,1
8,121.1012,9.418,generic,False,series spacing: 149.132-121.101=28.031 (ethylene),,,,,NaN,NaN,0
9,79.0543,9.327,generic,False,small loss: 97.065-79.054=18.011 (water),,,,,NaN,NaN,0
